# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Notebook chạy trọn bài lab trên **Google Colab T4**. Mã nằm trong thư mục `lab/` (không sửa `scripts/`, `tests/` của đề):

| File | Việc |
|---|---|
| `lab/common.py` | nhãn YOLO→JSON 0–1000, prompt, nạp model 4-bit, `encode_example` (mask loss), `collate`, `apply_setup` (LoRA/projector), suy luận |
| `lab/run.py` | chạy 1 thí nghiệm: huấn luyện → suy luận → `scripts/score.py` (tự làm tiếp nếu Colab ngắt) |
| `lab/inspect_model.py` | đếm tham số ViT / projector / LLM và số module LoRA (252 / 348) |
| `lab/reward.py`, `lab/grpo.py`, `lab/tests/` | E5: reward IoU + test + GRPO |
| `lab/report_tools.py` | bảng tổng hợp, ảnh minh hoạ đúng/sai, đường loss |

**Lần đầu:** mở notebook này từ GitHub trong Colab → chạy mục 0 (clone repo vào Drive) → từ lần sau mở `lab_day10.ipynb` **trong thư mục repo trên Drive** để thay đổi được lưu lại.
**Mỗi phiên mới:** chạy lại mục 0 và ô đầu của mục 1 (dữ liệu ở `/content` bị xoá khi hết phiên), rồi nhảy tới bước đang làm.
Các thí nghiệm chạy bằng `!python -m lab.run ...` trong tiến trình riêng nên VRAM được giải phóng sau mỗi lần; chạy lại đúng lệnh đó sau khi bị ngắt sẽ làm tiếp từ checkpoint.

## 0. Môi trường (Colab T4)

In [21]:
!git pull
!python -m lab.run --config configs/e0.yaml --name e0 --data-dir {DATA_DIR}

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.70 MiB | 3.29 MiB/s, done.
From https://github.com/Baor-AI/K4-Track-4-Day10-VisionLanguageModel
   b7fcf90..716bc09  main       -> origin/main
Updating c4c9322..716bc09
Fast-forward
 lab_day10.ipynb | 1524 ++++++++++++++++++++++++++++++++++++++-----------------
 1 file changed, 1049 insertions(+), 475 deletions(-)
/usr/bin/python3 /content/drive/MyDrive/K4-Day10/scripts/score.py --pred work/e0/pred_test.jsonl --name E0 --split test --coord norm1000 --out /content/drive/MyDrive/K4-Day10/report/evidence/e0.json --splits-dir /content/data/splits
E0 [test] n=311 valid=0.968 P=0.137 R=0.072 F1=0.094 CI95=0.068-0.123 meanIoU=0.606 FP/img=0.85
   pothole  P=0.196 R=0.074 F1=0.108
   crack    P=0.063 R=0.040 F1=0.049
   manhole  P=0.304 R=0.129 F1=0.181


In [22]:
# ==== BẠN ĐIỀN Ở ĐÂY ====
GITHUB_REPO = "https://github.com/Baor-AI/K4-Track-4-Day10-VisionLanguageModel"  # repo công khai của bạn
DRIVE_REPO  = "/content/drive/MyDrive/K4-Day10"   # repo + work/ (adapter, dự đoán) lưu trên Drive
DATA_DIR    = "/content/data"                     # dữ liệu ở ổ local của Colab (đọc nhanh hơn Drive)

# Tên các Secret trong Colab (biểu tượng chìa khoá bên trái). Giá trị KHÔNG ghi vào notebook.
SECRET_KAGGLE_USER = "KAGGLE_USERNAME"
SECRET_KAGGLE_KEY  = "KAGGLE_KEY"
SECRET_GH_TOKEN    = "GH_TOKEN"     # tuỳ chọn: GitHub token (quyền ghi repo) để push từ Colab
GIT_NAME  = "Nguyễn Văn Bảo"                      # tên hiển thị trong commit
GIT_EMAIL = "baour1210@gmail.com"                      # email commit

In [23]:
# Runtime → Change runtime type → T4 GPU
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists(f"{DRIVE_REPO}/.git"):
    !git clone {GITHUB_REPO} "{DRIVE_REPO}"
%cd "{DRIVE_REPO}"
!git config core.fileMode false
!git pull --ff-only || echo "bỏ qua pull (có thay đổi cục bộ)"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/K4-Day10
Already up to date.


In [ ]:
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!pip -q uninstall -y torchao
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

GPU 0: Tesla T4 (UUID: GPU-894fefb8-292e-feca-f49a-8af7f027889b)


In [ ]:
# Kaggle: dùng Colab Secrets. KHÔNG viết khoá vào notebook.
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get(SECRET_KAGGLE_USER)
os.environ['KAGGLE_KEY'] = userdata.get(SECRET_KAGGLE_KEY)

In [ ]:
# Công cụ của đề + test của bạn (không cần GPU)
!python -m pytest -q
!python -m pytest -q lab/tests

## 1. Dữ liệu: tải và chia split

In [ ]:
# Chạy lại mỗi phiên mới. Split cố định nên lần nào cũng ra cùng dataset_version.
if not os.path.exists(f"{DATA_DIR}/splits/meta.json"):
    !python scripts/prepare_data.py --kaggle --out {DATA_DIR} --work /content/kaggle_raw
# chép splits (nhỏ) vào repo để scripts/score.py và check_submission.py đối chiếu được (data/ đã .gitignore)
!mkdir -p data && rm -rf data/splits && cp -r {DATA_DIR}/splits data/splits
import json
meta = json.load(open(f"{DATA_DIR}/splits/meta.json"))
print("dataset_version:", meta["dataset_version"])

In [ ]:
from collections import Counter
from lab.common import load_split, open_image
splits = {s: load_split(s, DATA_DIR) for s in ("train", "val", "test")}
for s, rows in splits.items():
    c = Counter(o["label"] for r in rows for o in r["objects"])
    print(f"{s:5s} {len(rows):5d} ảnh  {sum(c.values()):5d} box  {dict(c)}  "
          f"ảnh không box: {sum(1 for r in rows if not r['objects'])}  "
          f"box nhiều nhất/ảnh: {max(len(r['objects']) for r in rows)}")
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
print("số phiên:", {s: len(v) for s, v in sess.items()})
print("phiên trùng train∩val, train∩test, val∩test:",
      len(sess["train"] & sess["val"]), len(sess["train"] & sess["test"]), len(sess["val"] & sess["test"]))

In [ ]:
# Vẽ 6 ảnh train có box (xanh lá = nhãn gốc)
import matplotlib.pyplot as plt, random
from lab.report_tools import draw
sample = random.Random(1).sample([r for r in splits["train"] if r["objects"]], 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, e in zip(axes.flat, sample):
    ax.imshow(draw(e, [], DATA_DIR)); ax.set_title(f"{e['image_id']}  {[o['label'] for o in e['objects']]}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab.common import yolo_to_box, objects_to_answer, make_prompt
e = sample[0]
ans = objects_to_answer(e["objects"])
print("PROMPT:", make_prompt()); print("ANSWER:", ans)
# Kiểm tra: JSON 0–1000 -> pixel phải trùng box YOLO gốc (sai số làm tròn ≤ 1 px, trừ phần bị kẹp ở biên)
W, H = e["width"], e["height"]
for o in e["objects"]:
    xc, yc, w, h = o["yolo"]
    gt_px = [(xc - w/2) * W, (yc - h/2) * H, (xc + w/2) * W, (yc + h/2) * H]
    b = yolo_to_box(o["label"], o["yolo"])["bbox_2d"]
    back = [b[0] / 1000 * W, b[1] / 1000 * H, b[2] / 1000 * W, b[3] / 1000 * H]
    print(o["label"], [round(v, 1) for v in gt_px], "->", b, "->", [round(v, 1) for v in back])

## 3. E0: zero-shot

In [ ]:
# Suy luận model gốc trên 311 ảnh test rồi chấm -> report/evidence/e0.json
!python -m lab.run --config configs/e0.yaml --name e0 --data-dir {DATA_DIR}

In [ ]:
# Xem output thô: model trả gì, có bọc ```json không, có hỏng JSON không?
from scripts.labcommon import read_jsonl
from scripts.score import parse_output
e0 = read_jsonl("work/e0/pred_test.jsonl")
for r in e0[:5]:
    print(r["image_id"], "|", r["raw"][:300].replace("\n", " "), "\n")
bad = [r for r in e0 if parse_output(r["raw"]) is None]
print(f"JSON hỏng: {len(bad)}/{len(e0)}")
for r in bad[:3]:
    print("HỎNG:", r["raw"][:300])
print(json.dumps(json.load(open("report/evidence/e0.json"))["micro"], indent=1))

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Kiểm tra trên CPU (chỉ tải processor): prompt là tiền tố, 220 token ảnh, chỉ phần trả lời có loss
from transformers import AutoProcessor
from lab.common import MODEL_ID, encode_example, check_template, collate
processor = AutoProcessor.from_pretrained(MODEL_ID)
prompt = make_prompt()
print("khớp chat template chính thức:", check_template(processor, prompt, ans))
ex = encode_example(processor, open_image(e, DATA_DIR), prompt, ans)
ids, labels = ex["input_ids"][0], ex["labels"][0]
img_tok = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
print({k: tuple(v.shape) for k, v in ex.items()})
print("token ảnh:", int((ids == img_tok).sum()), "| tổng token:", len(ids), "| token có loss:", int((labels != -100).sum()))
print("phần được học:", repr(processor.tokenizer.decode(ids[labels != -100])))
b = collate([ex, encode_example(processor, open_image(sample[1], DATA_DIR), prompt, objects_to_answer(sample[1]["objects"]))],
            processor.tokenizer.pad_token_id)
print({k: tuple(v.shape) for k, v in b.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
!pip -q uninstall -y torchao

In [ ]:
# Đếm tham số từng khối và số module LoRA (kỳ vọng 252 cho E1/E2, 348 cho E3; E2 hơn E1 ≈ 109M)
!python -m lab.inspect_model

## 6. Vòng huấn luyện
Mỗi lệnh: huấn luyện 300 bước (lưu checkpoint mỗi 100 bước vào `work/<tên>/` trên Drive) → suy luận test → chấm → `report/evidence/<tên>.json` (kèm `trainable_params`, `peak_vram_gb`, `train_minutes`).
Nếu Colab ngắt: chạy lại mục 0, ô đầu mục 1, rồi **chạy lại đúng lệnh đó**. Hết VRAM: thêm `--batch-size 1 --grad-accum 8`.

In [ ]:
# Chạy thử nhanh: 5 bước + 40 ảnh val với setup nặng nhất (E3) để chắc mọi thứ chạy (không tạo file bằng chứng)
!python -m lab.run --config configs/e3.yaml --name debug_e3 --max-steps 5 --split val --limit 40 --data-dir {DATA_DIR} --save-every 1000

In [ ]:
!python -m lab.run --config configs/e1.yaml --name e1 --data-dir {DATA_DIR}

In [ ]:
!python -m lab.run --config configs/e2.yaml --name e2 --data-dir {DATA_DIR}

In [ ]:
!python -m lab.run --config configs/e3.yaml --name e3 --data-dir {DATA_DIR}

## 7. Suy luận và chấm E1–E3

In [ ]:
# (Lệnh ở mục 6 đã suy luận + chấm.) Bảng tổng hợp để dán vào REPORT.md
from IPython.display import Markdown, display, Image as IPImage
from lab.report_tools import summary_table, plot_losses, save_examples
display(Markdown(summary_table()))
for n in ("e1", "e2", "e3"):
    p = f"work/{n}/run.json"
    if os.path.exists(p):
        r = json.load(open(p)); print(n, {k: r[k] for k in ("trainable_params", "peak_vram_gb", "train_minutes")})

In [ ]:
plot_losses(["e1", "e2", "e3"]); plt.show()

In [ ]:
# ≥ 6 ảnh minh hoạ đúng/sai (xanh lá = box đúng, màu khác = box dự đoán) -> report/figures/
for n in ("e0", "e1"):
    for p in save_examples(n, DATA_DIR):
        display(IPImage(str(p)))

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
Mặc định chọn biến **data_size** (rẻ nhất: giá trị 1.0 chính là setup gốc nên chỉ cần huấn luyện thêm 1 lần với 10% ảnh train).
Hai lựa chọn khác để sẵn trong ô dưới (bỏ comment nếu muốn), mỗi lựa chọn cần thêm 1–2 lần huấn luyện.

In [ ]:
BEST = "e1"   # <- đổi thành setup tốt nhất của bạn trong E1–E3 (xem bảng ở mục 7)
BEST_CFG = f"configs/{BEST}.yaml"

In [ ]:
# E4 data_size: 10% ảnh train (vẫn 300 bước) ...
!python -m lab.run --config {BEST_CFG} --name e4_data_size_0.1 --train-fraction 0.1 --label "E4 data_size=0.1" --data-dir {DATA_DIR}
# ... và 100% = chấm lại dự đoán của setup gốc dưới tên E4
from lab.run import score
run = json.load(open(f"work/{BEST}/run.json"))
score(f"work/{BEST}/pred_test.jsonl", "E4 data_size=1.0", "test", "norm1000",
      "report/evidence/e4_data_size_1.0.json", f"{DATA_DIR}/splits",
      {k: run[k] for k in ("trainable_params", "peak_vram_gb", "train_minutes")})

# --- Thay thế: rank (alpha = 2r) ---
# !python -m lab.run --config {BEST_CFG} --name e4_rank_4  --r 4  --label "E4 rank=4"  --data-dir {DATA_DIR}
# !python -m lab.run --config {BEST_CFG} --name e4_rank_64 --r 64 --label "E4 rank=64" --data-dir {DATA_DIR}
# --- Thay thế: coord_scale (đổi cả nhãn, prompt và --coord khi chấm) ---
# !python -m lab.run --config {BEST_CFG} --name e4_coord_scale_pixel --coord pixel --label "E4 coord=pixel" --data-dir {DATA_DIR}
#   norm1000 = setup gốc: chấm lại work/{BEST}/pred_test.jsonl như trên với tên e4_coord_scale_norm1000.json
display(Markdown(summary_table()))

In [ ]:
# E5 (tuỳ chọn, +10): test reward trước, rồi GRPO 60 bước từ setup tốt nhất -> report/evidence/e5.json
!python -m pytest -q lab/tests/test_reward.py
!python -m lab.grpo --base {BEST} --name e5 --data-dir {DATA_DIR}

In [ ]:
# Dấu hiệu reward hacking: tỉ lệ trả [] và diện tích box trung bình theo bước
g = read_jsonl("work/e5/grpo_log.jsonl")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a_, k in zip(ax, ("reward_mean", "empty_frac", "mean_box_area")):
    a_.plot([r["step"] for r in g], [r[k] for r in g]); a_.set_title(k); a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig("report/figures/grpo.png", dpi=120); plt.show()
display(Markdown(summary_table()))

## 9. Báo cáo và nộp bài

In [ ]:
# Điền REPORT.md (bảng ở mục 7 + 10 câu hỏi bằng số của bạn), lưu notebook (Ctrl+S), rồi kiểm tra:
!python scripts/check_submission.py

In [ ]:
# Commit + push từ Colab (cần Secret GH_TOKEN). Commit theo từng bước để đủ ≥ 5 commit.
import subprocess
def git_push(msg, paths=("lab", "report", "REPORT.md", "lab_day10.ipynb")):
    tok = userdata.get(SECRET_GH_TOKEN)
    subprocess.run(["git", "add", *paths], check=True)
    subprocess.run(["git", "-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}", "commit", "-m", msg])
    url = GITHUB_REPO.replace("https://", f"https://x-access-token:{tok}@")
    r = subprocess.run(["git", "push", url, "HEAD:main"], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(tok, "***"))
!git status --short
# git_push("E0: zero-shot baseline")

In [ ]:
git_push("E0 + E1: zero-shot và LoRA LLM")